In [115]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score,mean_squared_error
import matplotlib.pyplot as plt
%matplotlib inline

In [116]:
train = pd.read_csv('C:\\Users\\msi1\\Downloads\\Datasets\\Lesson 4\\Bigmart Datasets\\train.csv')
test = pd.read_csv('C:\\Users\\msi1\\Downloads\\Datasets\\Lesson 4\\Bigmart Datasets\\test.csv')

In [117]:
print(train.shape)
print(test.shape)

(8523, 12)
(5681, 11)


In [118]:
print(train.columns)
print(test.columns)

Index(['Item_Identifier', 'Item_Weight', 'Item_Fat_Content', 'Item_Visibility',
       'Item_Type', 'Item_MRP', 'Outlet_Identifier',
       'Outlet_Establishment_Year', 'Outlet_Size', 'Outlet_Location_Type',
       'Outlet_Type', 'Item_Outlet_Sales'],
      dtype='object')
Index(['Item_Identifier', 'Item_Weight', 'Item_Fat_Content', 'Item_Visibility',
       'Item_Type', 'Item_MRP', 'Outlet_Identifier',
       'Outlet_Establishment_Year', 'Outlet_Size', 'Outlet_Location_Type',
       'Outlet_Type'],
      dtype='object')


In [119]:
train['source'] = 'train'
test['source'] = 'test'


In [120]:
data = pd.concat([train,test],ignore_index = True)
data
print(data.shape)

(14204, 13)


In [121]:
data.head()

,Item_Fat_Content,Item_Identifier,Item_MRP,Item_Outlet_Sales,Item_Type,Item_Visibility,Item_Weight,Outlet_Establishment_Year,Outlet_Identifier,Outlet_Location_Type,Outlet_Size,Outlet_Type,source
0,Low Fat,FDA15,249.8092,3735.1380,Dairy,0.016047,9.30,1999,OUT049,Tier 1,Medium,Supermarket Type1,train
1,Regular,DRC01,48.2692,443.4228,Soft Drinks,0.019278,5.92,2009,OUT018,Tier 3,Medium,Supermarket Type2,train
2,Low Fat,FDN15,141.6180,2097.2700,Meat,0.016760,17.50,1999,OUT049,Tier 1,Medium,Supermarket Type1,train
3,Regular,FDX07,182.0950,732.3800,Fruits and Vegetables,0.000000,19.20,1998,OUT010,Tier 3,NaN,Grocery Store,train
4,Low Fat,NCD19,53.8614,994.7052,Household,0.000000,8.93,1987,OUT013,Tier 3,High,Supermarket Type1,train


In [122]:
data.describe()

,Item_MRP,Item_Outlet_Sales,Item_Visibility,Item_Weight,Outlet_Establishment_Year
count,14204.000000,8523.000000,14204.000000,11765.000000,14204.000000
mean,141.004977,2181.288914,0.065953,12.792854,1997.830681
std,62.086938,1706.499616,0.051459,4.652502,8.371664
min,31.290000,33.290000,0.000000,4.555000,1985.000000
25%,94.012000,834.247400,0.027036,8.710000,1987.000000
50%,142.247000,1794.331000,0.054021,12.600000,1999.000000
75%,185.855600,3101.296400,0.094037,16.750000,2004.000000
max,266.888400,13086.964800,0.328391,21.350000,2009.000000


In [123]:
data.isna().any()

Item_Fat_Content             False
Item_Identifier              False
Item_MRP                     False
Item_Outlet_Sales             True
Item_Type                    False
Item_Visibility              False
Item_Weight                   True
Outlet_Establishment_Year    False
Outlet_Identifier            False
Outlet_Location_Type         False
Outlet_Size                   True
Outlet_Type                  False
source                       False
dtype: bool

In [124]:
data.isnull().sum()

Item_Fat_Content                0
Item_Identifier                 0
Item_MRP                        0
Item_Outlet_Sales            5681
Item_Type                       0
Item_Visibility                 0
Item_Weight                  2439
Outlet_Establishment_Year       0
Outlet_Identifier               0
Outlet_Location_Type            0
Outlet_Size                  4016
Outlet_Type                     0
source                          0
dtype: int64

In [125]:
len(data['Item_Outlet_Sales'])

14204

In [126]:
len(data['Item_Weight'])

14204

In [127]:
data['Item_Outlet_Sales'].describe()

count     8523.000000
mean      2181.288914
std       1706.499616
min         33.290000
25%        834.247400
50%       1794.331000
75%       3101.296400
max      13086.964800
Name: Item_Outlet_Sales, dtype: float64

In [128]:
data['Item_Weight'].mode()

0    17.6
dtype: float64

In [129]:
data['Outlet_Size'].mode()

0    Medium
dtype: object

In [130]:
data['Outlet_Size'] = data['Outlet_Size'].fillna(data['Outlet_Size'].mode()[0])

In [131]:
data['Item_Weight'] = data['Item_Weight'].fillna(data['Item_Weight'].mean())

In [132]:
#outier detection and remove

Q1 = data['Item_Visibility'].quantile(0.25)
print(Q1)
Q3 = data['Item_Visibility'].quantile(0.75)
print(Q3)

0.0270356825
0.0940372535


In [133]:
IQR = Q3-Q1
IQR

0.067001571

In [134]:
new_data = data.query('(@Q1-1.5*@IQR)<=Item_Visibility<=(@Q3+1.5*@IQR)')
new_data.shape

data = new_data

In [135]:
data['Item_Visibility_bins'] = pd.cut(data['Item_Visibility'],[0.000,0.065,0.13,0.2], labels = ['low viz','viz','high viz'])
data['Item_Visibility_bins'].value_counts()

low viz     7363
viz         4283
high viz    1418
Name: Item_Visibility_bins, dtype: int64

In [136]:
data

,Item_Fat_Content,Item_Identifier,Item_MRP,Item_Outlet_Sales,Item_Type,Item_Visibility,Item_Weight,Outlet_Establishment_Year,Outlet_Identifier,Outlet_Location_Type,Outlet_Size,Outlet_Type,source,Item_Visibility_bins
0,Low Fat,FDA15,249.8092,3735.1380,Dairy,0.016047,9.30,1999,OUT049,Tier 1,Medium,Supermarket Type1,train,low viz
1,Regular,DRC01,48.2692,443.4228,Soft Drinks,0.019278,5.92,2009,OUT018,Tier 3,Medium,Supermarket Type2,train,low viz
2,Low Fat,FDN15,141.6180,2097.2700,Meat,0.016760,17.50,1999,OUT049,Tier 1,Medium,Supermarket Type1,train,low viz
3,Regular,FDX07,182.0950,732.3800,Fruits and Vegetables,0.000000,19.20,1998,OUT010,Tier 3,Medium,Grocery Store,train,NaN
4,Low Fat,NCD19,53.8614,994.7052,Household,0.000000,8.93,1987,OUT013,Tier 3,High,Supermarket Type1,train,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14199,Regular,FDB58,141.3154,NaN,Snack Foods,0.013496,10.50,1997,OUT046,Tier 1,Small,Supermarket Type1,test,low viz
14200,Regular,FDD47,169.1448,NaN,Starchy Foods,0.142991,7.60,2009,OUT018,Tier 3,Medium,Supermarket Type2,test,high viz
14201,Low Fat,NCO17,118.7440,NaN,Health and Hygiene,0.073529,10.00,2002,OUT045,Tier 2,Medium,Supermarket Type1,test,viz
14202,Regular,FDJ26,214.6218,NaN,Canned,0.000000,15.30,2007,OUT017,Tier 2,Medium,Supermarket Type1,test,NaN


In [137]:
#replace null value ffrom low viz
data['Item_Visibility_bins'] = data['Item_Visibility_bins'].replace(np.nan,'low viz',regex=True)

In [138]:
data['Item_Fat_Content'].value_counts()

Low Fat    8352
Regular    4721
LF          506
reg         190
low fat     174
Name: Item_Fat_Content, dtype: int64

In [139]:
data['Item_Fat_Content'] = data['Item_Fat_Content'].replace(['LF','low fat'],'Low Fat')

In [140]:
data['Item_Fat_Content'].value_counts()

Low Fat    9032
Regular    4721
reg         190
Name: Item_Fat_Content, dtype: int64

In [141]:
data['Item_Fat_Content'] = data['Item_Fat_Content'].replace('reg','Regular')

In [142]:
#convert all categorical variable to numeric

le = LabelEncoder()
data['Item_Fat_Content'] = le.fit_transform(data['Item_Fat_Content'])

In [143]:
data['Item_Visibility_bins'] = le.fit_transform(data['Item_Visibility_bins'])

In [144]:
data['Outlet_Size'] = le.fit_transform(data['Outlet_Size'])

In [145]:
data['Outlet_Location_Type'] = le.fit_transform(data['Outlet_Location_Type'])

In [146]:
data['Outlet_Type'].unique()

array(['Supermarket Type1', 'Supermarket Type2', 'Grocery Store',
       'Supermarket Type3'], dtype=object)

In [147]:
dummy = pd.get_dummies(data['Outlet_Type'])
dummy

,Grocery Store,Supermarket Type1,Supermarket Type2,Supermarket Type3
0,0,1,0,0
1,0,0,1,0
2,0,1,0,0
3,1,0,0,0
4,0,1,0,0
...,...,...,...,...
14199,0,1,0,0
14200,0,0,1,0
14201,0,1,0,0
14202,0,1,0,0


In [148]:
data['Item_Identifier'].unique()

array(['FDA15', 'DRC01', 'FDN15', ..., 'NCF55', 'NCW30', 'NCW05'],
      dtype=object)

In [149]:
data['Item_Identifier'].value_counts()

NCI06    10
FDJ58    10
FDT25    10
FDN50    10
DRC25    10
         ..
FDG21     7
FDH58     7
DRF51     7
FDM50     7
NCW54     7
Name: Item_Identifier, Length: 1559, dtype: int64

In [150]:
#multiple category present in Item identifier we can reduce by using mapping
data['Item_Type_Combined'] = data['Item_Identifier'].apply(lambda x: x[0:2])
data['Item_Type_Combined'] = data['Item_Type_Combined'].map({'FD': 'Food','NC': 'Non-Consumable','DR': 'Drinks'})

In [151]:
data['Item_Type_Combined'].value_counts()

Food              9991
Non-Consumable    2652
Drinks            1300
Name: Item_Type_Combined, dtype: int64

In [152]:
data.shape

(13943, 15)

In [153]:
data

,Item_Fat_Content,Item_Identifier,Item_MRP,Item_Outlet_Sales,Item_Type,Item_Visibility,Item_Weight,Outlet_Establishment_Year,Outlet_Identifier,Outlet_Location_Type,Outlet_Size,Outlet_Type,source,Item_Visibility_bins,Item_Type_Combined
0,0,FDA15,249.8092,3735.1380,Dairy,0.016047,9.30,1999,OUT049,0,1,Supermarket Type1,train,1,Food
1,1,DRC01,48.2692,443.4228,Soft Drinks,0.019278,5.92,2009,OUT018,2,1,Supermarket Type2,train,1,Drinks
2,0,FDN15,141.6180,2097.2700,Meat,0.016760,17.50,1999,OUT049,0,1,Supermarket Type1,train,1,Food
3,1,FDX07,182.0950,732.3800,Fruits and Vegetables,0.000000,19.20,1998,OUT010,2,1,Grocery Store,train,1,Food
4,0,NCD19,53.8614,994.7052,Household,0.000000,8.93,1987,OUT013,2,0,Supermarket Type1,train,1,Non-Consumable
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14199,1,FDB58,141.3154,NaN,Snack Foods,0.013496,10.50,1997,OUT046,0,2,Supermarket Type1,test,1,Food
14200,1,FDD47,169.1448,NaN,Starchy Foods,0.142991,7.60,2009,OUT018,2,1,Supermarket Type2,test,0,Food
14201,0,NCO17,118.7440,NaN,Health and Hygiene,0.073529,10.00,2002,OUT045,1,1,Supermarket Type1,test,2,Non-Consumable
14202,1,FDJ26,214.6218,NaN,Canned,0.000000,15.30,2007,OUT017,1,1,Supermarket Type1,test,1,Food


In [154]:
#perform one-hot encoding to the categorical columns

data = pd.get_dummies(data,columns = ['Item_Fat_Content','Item_Type_Combined','Outlet_Type','Outlet_Size','Outlet_Location_Type'])
data

,Item_Identifier,Item_MRP,Item_Outlet_Sales,Item_Type,Item_Visibility,Item_Weight,Outlet_Establishment_Year,Outlet_Identifier,source,Item_Visibility_bins,...,Outlet_Type_Grocery Store,Outlet_Type_Supermarket Type1,Outlet_Type_Supermarket Type2,Outlet_Type_Supermarket Type3,Outlet_Size_0,Outlet_Size_1,Outlet_Size_2,Outlet_Location_Type_0,Outlet_Location_Type_1,Outlet_Location_Type_2
0,FDA15,249.8092,3735.1380,Dairy,0.016047,9.30,1999,OUT049,train,1,...,0,1,0,0,0,1,0,1,0,0
1,DRC01,48.2692,443.4228,Soft Drinks,0.019278,5.92,2009,OUT018,train,1,...,0,0,1,0,0,1,0,0,0,1
2,FDN15,141.6180,2097.2700,Meat,0.016760,17.50,1999,OUT049,train,1,...,0,1,0,0,0,1,0,1,0,0
3,FDX07,182.0950,732.3800,Fruits and Vegetables,0.000000,19.20,1998,OUT010,train,1,...,1,0,0,0,0,1,0,0,0,1
4,NCD19,53.8614,994.7052,Household,0.000000,8.93,1987,OUT013,train,1,...,0,1,0,0,1,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14199,FDB58,141.3154,NaN,Snack Foods,0.013496,10.50,1997,OUT046,test,1,...,0,1,0,0,0,0,1,1,0,0
14200,FDD47,169.1448,NaN,Starchy Foods,0.142991,7.60,2009,OUT018,test,0,...,0,0,1,0,0,1,0,0,0,1
14201,NCO17,118.7440,NaN,Health and Hygiene,0.073529,10.00,2002,OUT045,test,2,...,0,1,0,0,0,1,0,0,1,0
14202,FDJ26,214.6218,NaN,Canned,0.000000,15.30,2007,OUT017,test,1,...,0,1,0,0,0,1,0,0,1,0


In [155]:
data.dtypes

Item_Identifier                       object
Item_MRP                             float64
Item_Outlet_Sales                    float64
Item_Type                             object
Item_Visibility                      float64
Item_Weight                          float64
Outlet_Establishment_Year              int64
Outlet_Identifier                     object
source                                object
Item_Visibility_bins                   int32
Item_Fat_Content_0                     uint8
Item_Fat_Content_1                     uint8
Item_Type_Combined_Drinks              uint8
Item_Type_Combined_Food                uint8
Item_Type_Combined_Non-Consumable      uint8
Outlet_Type_Grocery Store              uint8
Outlet_Type_Supermarket Type1          uint8
Outlet_Type_Supermarket Type2          uint8
Outlet_Type_Supermarket Type3          uint8
Outlet_Size_0                          uint8
Outlet_Size_1                          uint8
Outlet_Size_2                          uint8
Outlet_Loc

In [161]:
import warnings
warnings.filterwarnings('ignore')


#data.drop(['Item_Type','Outlet_Establishment_Year'], axis=1,inplace = True)

train = data.loc[data['source'] == 'train']
test = data.loc[data['source'] == 'test' ]
train

,Item_Identifier,Item_MRP,Item_Outlet_Sales,Item_Type,Item_Visibility,Item_Weight,Outlet_Establishment_Year,Outlet_Identifier,source,Item_Visibility_bins,...,Outlet_Type_Grocery Store,Outlet_Type_Supermarket Type1,Outlet_Type_Supermarket Type2,Outlet_Type_Supermarket Type3,Outlet_Size_0,Outlet_Size_1,Outlet_Size_2,Outlet_Location_Type_0,Outlet_Location_Type_1,Outlet_Location_Type_2
0,FDA15,249.8092,3735.1380,Dairy,0.016047,9.300,1999,OUT049,train,1,...,0,1,0,0,0,1,0,1,0,0
1,DRC01,48.2692,443.4228,Soft Drinks,0.019278,5.920,2009,OUT018,train,1,...,0,0,1,0,0,1,0,0,0,1
2,FDN15,141.6180,2097.2700,Meat,0.016760,17.500,1999,OUT049,train,1,...,0,1,0,0,0,1,0,1,0,0
3,FDX07,182.0950,732.3800,Fruits and Vegetables,0.000000,19.200,1998,OUT010,train,1,...,1,0,0,0,0,1,0,0,0,1
4,NCD19,53.8614,994.7052,Household,0.000000,8.930,1987,OUT013,train,1,...,0,1,0,0,1,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8518,FDF22,214.5218,2778.3834,Snack Foods,0.056783,6.865,1987,OUT013,train,1,...,0,1,0,0,1,0,0,0,0,1
8519,FDS36,108.1570,549.2850,Baking Goods,0.046982,8.380,2002,OUT045,train,1,...,0,1,0,0,0,1,0,0,1,0
8520,NCJ29,85.1224,1193.1136,Health and Hygiene,0.035186,10.600,2004,OUT035,train,1,...,0,1,0,0,0,0,1,0,1,0
8521,FDN46,103.1332,1845.5976,Snack Foods,0.145221,7.210,2009,OUT018,train,0,...,0,0,1,0,0,1,0,0,0,1


In [162]:
test.drop(['Item_Outlet_Sales','source'],axis =1, inplace = True)

In [163]:
train.drop(['source'],axis = 1,inplace = True)

In [165]:
train.to_csv('data_modify.csv',index = False)
test.to_csv('test_modify.csv',index = False)

In [167]:
train2 = pd.read_csv('data_modify.csv')
train2

,Item_Identifier,Item_MRP,Item_Outlet_Sales,Item_Type,Item_Visibility,Item_Weight,Outlet_Establishment_Year,Outlet_Identifier,Item_Visibility_bins,Item_Fat_Content_0,...,Outlet_Type_Grocery Store,Outlet_Type_Supermarket Type1,Outlet_Type_Supermarket Type2,Outlet_Type_Supermarket Type3,Outlet_Size_0,Outlet_Size_1,Outlet_Size_2,Outlet_Location_Type_0,Outlet_Location_Type_1,Outlet_Location_Type_2
0,FDA15,249.8092,3735.1380,Dairy,0.016047,9.300,1999,OUT049,1,1,...,0,1,0,0,0,1,0,1,0,0
1,DRC01,48.2692,443.4228,Soft Drinks,0.019278,5.920,2009,OUT018,1,0,...,0,0,1,0,0,1,0,0,0,1
2,FDN15,141.6180,2097.2700,Meat,0.016760,17.500,1999,OUT049,1,1,...,0,1,0,0,0,1,0,1,0,0
3,FDX07,182.0950,732.3800,Fruits and Vegetables,0.000000,19.200,1998,OUT010,1,0,...,1,0,0,0,0,1,0,0,0,1
4,NCD19,53.8614,994.7052,Household,0.000000,8.930,1987,OUT013,1,1,...,0,1,0,0,1,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8370,FDF22,214.5218,2778.3834,Snack Foods,0.056783,6.865,1987,OUT013,1,1,...,0,1,0,0,1,0,0,0,0,1
8371,FDS36,108.1570,549.2850,Baking Goods,0.046982,8.380,2002,OUT045,1,0,...,0,1,0,0,0,1,0,0,1,0
8372,NCJ29,85.1224,1193.1136,Health and Hygiene,0.035186,10.600,2004,OUT035,1,1,...,0,1,0,0,0,0,1,0,1,0
8373,FDN46,103.1332,1845.5976,Snack Foods,0.145221,7.210,2009,OUT018,0,0,...,0,0,1,0,0,1,0,0,0,1


In [169]:
train2.dtypes

Item_Identifier                       object
Item_MRP                             float64
Item_Outlet_Sales                    float64
Item_Type                             object
Item_Visibility                      float64
Item_Weight                          float64
Outlet_Establishment_Year              int64
Outlet_Identifier                     object
Item_Visibility_bins                   int64
Item_Fat_Content_0                     int64
Item_Fat_Content_1                     int64
Item_Type_Combined_Drinks              int64
Item_Type_Combined_Food                int64
Item_Type_Combined_Non-Consumable      int64
Outlet_Type_Grocery Store              int64
Outlet_Type_Supermarket Type1          int64
Outlet_Type_Supermarket Type2          int64
Outlet_Type_Supermarket Type3          int64
Outlet_Size_0                          int64
Outlet_Size_1                          int64
Outlet_Size_2                          int64
Outlet_Location_Type_0                 int64
Outlet_Loc

In [170]:
X_train = train2.drop(['Item_Outlet_Sales','Item_Identifier','Outlet_Identifier','Item_Type'],axis = 1)
y_train = train2.Item_Outlet_Sales

In [172]:
test2 = pd.read_csv('test_modify.csv')
X_test = test2.drop(['Outlet_Identifier','Item_Identifier','Item_Type'],axis=1)
X_test

,Item_MRP,Item_Visibility,Item_Weight,Outlet_Establishment_Year,Item_Visibility_bins,Item_Fat_Content_0,Item_Fat_Content_1,Item_Type_Combined_Drinks,Item_Type_Combined_Food,Item_Type_Combined_Non-Consumable,Outlet_Type_Grocery Store,Outlet_Type_Supermarket Type1,Outlet_Type_Supermarket Type2,Outlet_Type_Supermarket Type3,Outlet_Size_0,Outlet_Size_1,Outlet_Size_2,Outlet_Location_Type_0,Outlet_Location_Type_1,Outlet_Location_Type_2
0,107.8622,0.007565,20.750000,1999,1,1,0,0,1,0,0,1,0,0,0,1,0,1,0,0
1,87.3198,0.038428,8.300000,2007,1,0,1,0,1,0,0,1,0,0,0,1,0,0,1,0
2,241.7538,0.099575,14.600000,1998,2,1,0,0,0,1,1,0,0,0,0,1,0,0,0,1
3,155.0340,0.015388,7.315000,2007,1,1,0,0,1,0,0,1,0,0,0,1,0,0,1,0
4,234.2300,0.118599,12.792854,1985,2,0,1,0,1,0,0,0,0,1,0,1,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5563,141.3154,0.013496,10.500000,1997,1,0,1,0,1,0,0,1,0,0,0,0,1,1,0,0
5564,169.1448,0.142991,7.600000,2009,0,0,1,0,1,0,0,0,1,0,0,1,0,0,0,1
5565,118.7440,0.073529,10.000000,2002,2,1,0,0,0,1,0,1,0,0,0,1,0,0,1,0
5566,214.6218,0.000000,15.300000,2007,1,0,1,0,1,0,0,1,0,0,0,1,0,0,1,0


In [175]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X_train,y_train,test_size = 0.3,random_state = 42)

In [176]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train,y_train)

LinearRegression(copy_X=True, fit_intercept=True, n_jobs=None, normalize=False)

In [177]:
lr.coef_

array([ 1.58826212e+01, -3.32746294e+02, -1.92692654e+00,  3.19549640e+01,
       -1.41755050e+01, -1.45901761e+00,  1.45901761e+00,  8.83932211e+00,
        3.46875666e+01, -4.35268887e+01, -1.63576909e+03, -1.25872012e+02,
       -3.48803423e+02,  2.11044453e+03,  5.39642616e+02, -3.11411308e+02,
       -2.28231308e+02,  1.88878689e+02,  4.79867957e+01, -2.36865485e+02])

In [179]:
prediction = lr.predict(X_test)

In [182]:
import math
print(math.sqrt(mean_squared_error(y_test,prediction)))

1126.4071066411786
